In [1]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
from sksurv.util import Surv
from src.data import load_dataset, split_data

df = load_dataset("telco")
X_train, X_test, y_train, y_test = split_data(df)

def to_surv(X, y):
    time = X["tenure"].clip(lower=0.5)      # 11 customers have tenure 0; give them half a month
    return Surv.from_arrays(event=y.astype(bool), time=time)

def predictors(X):
    return X.drop(columns=["tenure", "TotalCharges"])

S_train, S_test = to_surv(X_train, y_train), to_surv(X_test, y_test)
P_train, P_test = predictors(X_train), predictors(X_test)

print("tenure = 0 rows (train):", int((X_train["tenure"] == 0).sum()))
print("predictors:", P_train.shape[1], "| events:", int(y_train.sum()), "of", len(y_train))
print("median time, churners:", float(np.median(S_train["time"][S_train["event"]])))
print("max time:", float(S_train["time"].max()))

tenure = 0 rows (train): 8
predictors: 17 | events: 1495 of 5634
median time, churners: 10.0
max time: 72.0


In [2]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sksurv.linear_model import CoxPHSurvivalAnalysis
from src.models import make_preprocessor

def cv_cindex(make_model, P, S, n=5):
    scores = []
    for tr, va in StratifiedKFold(n, shuffle=True, random_state=42).split(P, S["event"]):
        m = make_model(P).fit(P.iloc[tr], S[tr])
        scores.append(m.score(P.iloc[va], S[va]))   # score = concordance index
    return round(np.mean(scores), 3), round(np.std(scores), 3)

make_cox = lambda P: Pipeline([("prep", make_preprocessor(P)),
                               ("model", CoxPHSurvivalAnalysis(alpha=0.1))])

print("Cox C-index (mean, sd):", cv_cindex(make_cox, P_train, S_train))

Cox C-index (mean, sd): (np.float64(0.867), np.float64(0.003))


In [3]:
from sksurv.ensemble import RandomSurvivalForest

make_rsf = lambda P: Pipeline([("prep", make_preprocessor(P)),
                               ("model", RandomSurvivalForest(n_estimators=200, min_samples_leaf=15,
                                                              n_jobs=-1, random_state=42))])

print("RSF C-index (mean, sd):", cv_cindex(make_rsf, P_train, S_train))

RSF C-index (mean, sd): (np.float64(0.852), np.float64(0.004))


In [4]:
import xgboost as xgb
from sksurv.metrics import concordance_index_censored

AFT_PARAMS = dict(objective="survival:aft", eval_metric="aft-nloglik",
                  aft_loss_distribution="normal", aft_loss_distribution_scale=1.2,
                  learning_rate=0.05, max_depth=3, subsample=0.8, colsample_bytree=0.8, seed=42)

def aft_matrix(X, S=None):
    d = xgb.DMatrix(X)
    if S is not None:                       # censored customers: true time is at least "time" (upper bound = infinity)
        d.set_float_info("label_lower_bound", S["time"])
        d.set_float_info("label_upper_bound", np.where(S["event"], S["time"], np.inf))
    return d

scores = []
for tr, va in StratifiedKFold(5, shuffle=True, random_state=42).split(P_train, S_train["event"]):
    prep = make_preprocessor(P_train)
    Xtr, Xva = prep.fit_transform(P_train.iloc[tr]), prep.transform(P_train.iloc[va])
    bst = xgb.train(AFT_PARAMS, aft_matrix(Xtr, S_train[tr]), num_boost_round=300)
    pred_time = bst.predict(aft_matrix(Xva))            # predicted lifetime
    Sva = S_train[va]
    scores.append(concordance_index_censored(Sva["event"], Sva["time"], -pred_time)[0])  # longer life = lower risk

print("XGBoost-AFT C-index (mean, sd):", round(np.mean(scores), 3), round(np.std(scores), 3))

XGBoost-AFT C-index (mean, sd): 0.865 0.004


In [5]:
from scipy.stats import norm
from sksurv.metrics import integrated_brier_score
from sksurv.nonparametric import kaplan_meier_estimator
from sksurv.ensemble import RandomSurvivalForest

TIMES = np.arange(3, 61, 3)          # months at which the Brier score is evaluated

def surv_matrix(model, X):
    return np.asarray([[fn(t) for t in TIMES] for fn in model.predict_survival_function(X)])

ibs = {"No features (Kaplan-Meier)": [], "Cox": [], "RSF": [], "XGBoost-AFT": []}
for tr, va in StratifiedKFold(5, shuffle=True, random_state=42).split(P_train, S_train["event"]):
    prep = make_preprocessor(P_train)
    Xtr, Xva = prep.fit_transform(P_train.iloc[tr]), prep.transform(P_train.iloc[va])
    Str, Sva = S_train[tr], S_train[va]
    est = {}

    t, s = kaplan_meier_estimator(Str["event"], Str["time"])
    est["No features (Kaplan-Meier)"] = np.tile(s[np.searchsorted(t, TIMES, side="right") - 1], (len(Sva), 1))

    est["Cox"] = surv_matrix(CoxPHSurvivalAnalysis(alpha=0.1).fit(Xtr, Str), Xva)
    est["RSF"] = surv_matrix(RandomSurvivalForest(n_estimators=200, min_samples_leaf=15,
                                                  n_jobs=-1, random_state=42).fit(Xtr, Str), Xva)

    bst = xgb.train(AFT_PARAMS, aft_matrix(Xtr, Str), num_boost_round=300)
    pred = bst.predict(aft_matrix(Xva))             # predicted lifetime
    est["XGBoost-AFT"] = 1 - norm.cdf((np.log(TIMES)[None, :] - np.log(pred)[:, None]) / 1.2)

    for name, e in est.items():
        ibs[name].append(integrated_brier_score(Str, Sva, e, TIMES))

for name, v in ibs.items():
    print(f"{name:28s} IBS mean {np.mean(v):.3f}  sd {np.std(v):.3f}")

No features (Kaplan-Meier)   IBS mean 0.175  sd 0.001
Cox                          IBS mean 0.084  sd 0.002
RSF                          IBS mean 0.092  sd 0.003
XGBoost-AFT                  IBS mean 0.082  sd 0.002


In [6]:
H = 12   # months ahead

def expected_months(model, X, tenure, H=H):
    """Expected number of the next H months a customer is still active, given the tenure so far."""
    out = []
    for fn, t0 in zip(model.predict_survival_function(X), tenure):
        lo, hi = fn.domain
        s0 = fn(min(max(t0, lo), hi))
        s = [fn(min(t0 + k, hi)) for k in range(1, H + 1)]
        out.append(np.sum(s) / s0)
    return np.array(out)

months = pd.Series(np.nan, index=P_train.index)
for tr, va in StratifiedKFold(5, shuffle=True, random_state=42).split(P_train, S_train["event"]):
    prep = make_preprocessor(P_train)
    Xtr, Xva = prep.fit_transform(P_train.iloc[tr]), prep.transform(P_train.iloc[va])
    cox = CoxPHSurvivalAnalysis(alpha=0.1).fit(Xtr, S_train[tr])
    months.iloc[va] = expected_months(cox, Xva, X_train["tenure"].iloc[va].values)

V_i = months * X_train["MonthlyCharges"]
print("mean expected months (max 12):", round(months.mean(), 2))
print("mean V_i:", round(V_i.mean(), 1), "| fixed V was 777")
print(V_i.describe().round(1))
print(V_i.groupby(X_train["Contract"]).mean().round(1))
print(V_i.groupby(y_train).mean().round(1))      # 0 = stayed, 1 = churned

pd.DataFrame({"expected_months": months, "V_i": V_i}).to_csv("../results/lifetime_value_telco.csv")

mean expected months (max 12): 10.74
mean V_i: 688.8 | fixed V was 777
count    5634.0
mean      688.8
std       335.3
min        40.8
25%       365.6
50%       697.6
75%       952.3
max      1425.0
dtype: float64
Contract
Month-to-month    653.8
One year          741.9
Two year          723.0
dtype: float64
churn
0    669.8
1    741.4
dtype: float64


In [7]:
from src.evaluate import oof_proba
from src.models import make_xgb

proba = oof_proba(make_xgb(X_train), X_train, y_train)     # same honest out-of-fold probabilities as before
y = y_train.values.astype(bool)
offer, s = 78, 0.3

def avg_cost(flag, V):
    flagged = offer + np.where(y, (1 - s) * V, 0)      # offer paid; if a real leaver, 30% are saved
    unflagged = np.where(y, V, 0)                      # a missed leaver costs V
    return float(np.mean(np.where(flag, flagged, unflagged)))

grid = np.arange(0.05, 0.96, 0.01)
rows = []
for label, V in [("fixed V = 777", np.full(len(y), 777.0)), ("customer-specific V_i", V_i.values)]:
    nobody = avg_cost(np.zeros(len(y), bool), V)
    costs = [avg_cost(proba >= t, V) for t in grid]
    rule_flag = proba >= offer / (s * V)               # flag if risk > this customer's own break-even
    rows.append({"setting": label,
                 "flag_nobody": nobody,
                 "global_thr_0.36": avg_cost(proba >= 0.36, V),
                 "best_global_thr": round(grid[int(np.argmin(costs))], 2),
                 "best_global_cost": min(costs),
                 "per_customer_rule": avg_cost(rule_flag, V),
                 "n_flagged_by_rule": int(rule_flag.sum())})

out = pd.DataFrame(rows).set_index("setting")
out["saving_global_%"] = ((1 - out["best_global_cost"] / out["flag_nobody"]) * 100).round(1)
out["saving_rule_%"] = ((1 - out["per_customer_rule"] / out["flag_nobody"]) * 100).round(1)
out.round(2).to_csv("../results/cost_lifetime_value_telco.csv")
out.round(2).T

setting,fixed V = 777,customer-specific V_i
flag_nobody,206.18,196.74
global_thr_0.36,188.11,180.06
best_global_thr,0.36,0.41
best_global_cost,188.11,179.60
per_customer_rule,188.54,178.32
n_flagged_by_rule,1897.00,1673.00
saving_global_%,8.80,8.70
saving_rule_%,8.60,9.40
